# Rebuild the Trend raster tiles with a continuous colormap (for GitHub Pages)

This regenerates the `trend` raster layer (the 2017-2025 EI slope layer on
the dashboard) so its actual pixel colors are a genuinely continuous
gradient from Decrease to Increase, with **zero as the exact neutral
midpoint** -- addressing the client comment directly (currently it reads
as a binary two-color classification because the old colormap's
white/neutral band was far narrower than the data's real spread, so
almost every pixel fell solidly into one extreme color or the other).

Run this in your `soilhealth` conda env (the one with GDAL already
installed -- same one used for the ORP/kraj spatial join).

Steps:
1. Reproject the slope raster to EPSG:3857 (Web Mercator) if you don't
   already have that version -- `gdal2tiles.py --xyz -p mercator` needs
   the source in that CRS.
2. Write a 3-stop colormap file: orange at -0.0169, white at exactly
   0, dark green at +0.0169 (roughly +/-2 standard deviations of the
   real data), with anything more extreme clipping to solid color --
   so the middle of the distribution actually shows visible contrast
   instead of washing out to near-white.
3. Run `gdaldem color-relief` to bake those colors into the raster.
4. Run `gdal2tiles.py` to produce the `{z}/{x}/{y}.png` tile pyramid
   (zoom 0-12, matching what the dashboard actually requests).
5. Push the `tiles/` folder to your (new) GitHub Pages repo at the same
   path the dashboard already points to -- no dashboard code changes
   needed once it's live at the same URL.

In [ ]:
import subprocess
from pathlib import Path

# --- adjust these paths if yours differ ---
SLOPE_TIF = Path("/Users/alonso.gonzalez.glez/Desktop/Instrumentalization/Outputs/Trend/EI_trend_slope_2017_2025.tif")
WORK_DIR = Path("/Users/alonso.gonzalez.glez/Desktop/Instrumentalization/Outputs/Trend")
TILES_OUT = WORK_DIR / "tiles"

SLOPE_3857 = WORK_DIR / "EI_trend_slope_2017_2025_3857.tif"
COLORED_TIF = WORK_DIR / "EI_trend_colored.tif"
COLORMAP_TXT = WORK_DIR / "trend_colormap.txt"

assert SLOPE_TIF.exists(), f"Can't find {SLOPE_TIF} -- update SLOPE_TIF above"
print("Source slope raster:", SLOPE_TIF)

In [ ]:
def run(cmd):
    """Run a shell command, streaming output, and raise if it fails."""
    print("$", " ".join(str(c) for c in cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"Command failed (exit {result.returncode}): {cmd[0]}")
    return result

In [ ]:
# Step 1 -- reproject to EPSG:3857 if not already done.
# Skips the reprojection if EI_trend_slope_2017_2025_3857.tif already exists
# from a previous run -- delete it first if you want to force a rebuild.
if not SLOPE_3857.exists():
    run(["gdalwarp", "-t_srs", "EPSG:3857", "-r", "bilinear", str(SLOPE_TIF), str(SLOPE_3857)])
else:
    print("Reprojected raster already exists, skipping gdalwarp:", SLOPE_3857)

In [ ]:
# Step 2 -- write the colormap file.
#
# A plain linear stretch across the full -0.072 to +0.054 range (tried
# previously) technically puts zero at the midpoint, but this raster's
# real values barely move relative to that range (mean -0.0042, std
# 0.0085) -- so almost every pixel ends up pale near-white and the map
# reads as washed out, even though the color logic is correct.
#
# Fix: keep the *labels* honest about the true range, but clip full
# color saturation at +/-2 standard deviations (~0.0169) instead of the
# extreme min/max. GDAL colors anything past the last defined stop with
# that stop's color, so this doesn't break continuity -- it just means
# roughly the middle 95% of pixels (a normal-ish spread) get real,
# visible gradation, and only the most extreme ~5% clip to solid orange
# or green. Zero is still the exact color midpoint.
colormap_text = """-0.01694  229 137 84
0.00000  253 253 253
0.01694  0 68 27
nv 255 255 255 0
"""
COLORMAP_TXT.write_text(colormap_text)
print("Wrote colormap:", COLORMAP_TXT)
print(colormap_text)

In [ ]:
# Step 3 -- bake the colors into the raster (RGBA output, -alpha keeps
# nodata pixels transparent instead of black).
run(["gdaldem", "color-relief", str(SLOPE_3857), str(COLORMAP_TXT), str(COLORED_TIF), "-alpha"])
print("Colored raster:", COLORED_TIF)

In [ ]:
# Step 4 -- generate the {z}/{x}/{y}.png tile pyramid.
# Same zoom range (0-12) and XYZ/mercator scheme as before, so the
# dashboard's tileLayer URL keeps working unchanged once these are
# pushed to the new repo.
if TILES_OUT.exists():
    print(f"NOTE: {TILES_OUT} already exists -- gdal2tiles.py will merge/overwrite into it.")
run(["gdal2tiles.py", "--xyz", "-p", "mercator", "-z", "0-12", str(COLORED_TIF), str(TILES_OUT)])
print("Tiles written to:", TILES_OUT)

## Step 5 -- publish

Push the contents of `tiles/` to your new GitHub Pages repo at the same
path the dashboard already points to
(`.../tiles/{z}/{x}/{y}.png`):

```bash
cp -r /Users/alonso.gonzalez.glez/Desktop/Instrumentalization/Outputs/Trend/tiles/* /path/to/your/new-github-pages-repo/tiles/
cd /path/to/your/new-github-pages-repo
git add tiles/
git commit -m "Recolor trend raster tiles with continuous gradient, zero-centered"
git push
```

If the new repo's Pages URL is different from the old one
(`alonsonetmxorg-web.github.io/...`), send me the new URL so it can be
swapped into `dashboard_logic.js` (`RASTER_TILE_URLS.trend`) and the
dashboard rebuilt/redeployed -- if it's the exact same URL, nothing on
the dashboard side needs to change once GitHub Pages redeploys.

The dashboard's own legend title bug ("Index Index" showing twice) and
the softened legend gradient bar have already been fixed and deployed
separately -- once these new tiles are live, the raster should finally
match what the legend has been showing all along.